In [5]:
import cv2
import numpy as np
import requests

def extrair_caracteristicas_cor_e_intensidade(url_imagem):
    """
    Baixa uma imagem da URL e extrai um vetor de características baseado 
    no histograma RGB e na intensidade da imagem.
    """
    try:
        # 1. Baixa a imagem da web
        headers = {"User-Agent": "Mozilla/5.0"}
        resposta = requests.get(url_imagem, headers=headers, stream=True, timeout=10)
        if resposta.status_code != 200:
            print(f"Erro ao acessar a URL: HTTP {resposta.status_code}")
            return None
            
        # Converte os bytes da resposta em uma imagem legível pelo OpenCV
        imagem_array = np.asarray(bytearray(resposta.content), dtype=np.uint8)
        img = cv2.imdecode(imagem_array, cv2.IMREAD_COLOR) 
        
        if img is None:
            print("Não foi possível decodificar a imagem.")
            return None

        # 2. Redimensiona para padronizar (150x150 pixels)
        img = cv2.resize(img, (150, 150))
        
        # 3. Separa os canais de cores (O OpenCV lê como BGR)
        b_canal, g_canal, r_canal = cv2.split(img)
        
        # 4. Calcula o Histograma para cada cor (8 faixas/baldes por cor)
        bins = 8
        hist_r = cv2.calcHist([r_canal], [0], None, [bins], [0, 256])
        hist_g = cv2.calcHist([g_canal], [0], None, [bins], [0, 256])
        hist_b = cv2.calcHist([b_canal], [0], None, [bins], [0, 256])
        
        # Normaliza para que os valores fiquem comparáveis
        hist_r = cv2.normalize(hist_r, hist_r).flatten()
        hist_g = cv2.normalize(hist_g, hist_g).flatten()
        hist_b = cv2.normalize(hist_b, hist_b).flatten()
        
        # 5. Calcula a Intensidade (Luminosidade) em tons de cinza
        img_cinza = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        media_intensidade = np.mean(img_cinza) / 255.0  
        desvio_intensidade = np.std(img_cinza) / 255.0  
        
        # 6. Une tudo em um vetor descritivo "caixa aberta" de 26 números
        vetor = np.concatenate([
            hist_r, 
            hist_g, 
            hist_b, 
            [media_intensidade, desvio_intensidade]
        ])
        return vetor

    except Exception as e:
        print(f"Falha no processamento: {e}")
        return None

# ==========================================
# TESTE RÁPIDO COM DOIS LINKS DO SEU BANCO
# ==========================================

# Pegando os links de imagem reais do seu arquivo limpo
url_frieren = "https://cdn.myanimelist.net/images/anime/1171/156397.jpg"

# IMPORTANTE: Mudei a segunda URL para a capa de 'Solo Leveling' real do seu banco 
# para você ver a diferença matemática acontecer!
url_solo_leveling = "https://cdn.myanimelist.net/images/anime/1540/155824.jpg" 

print("Processando imagem do Anime 1...")
vetor_anime1 = extrair_caracteristicas_cor_e_intensidade(url_frieren)

print("Processando imagem do Anime 2...")
vetor_anime2 = extrair_caracteristicas_cor_e_intensidade(url_solo_leveling)

# Se ambos os vetores foram gerados com sucesso, calcula a distância
if vetor_anime1 is not None and vetor_anime2 is not None:
    # CALCULO DA DISTÂNCIA EUCLIDIANA USANDO APENAS NUMPY (MUITO MAIS RÁPIDO)
    distancia = np.linalg.norm(vetor_anime1 - vetor_anime2)
    
    print("\n--- RESULTADO DO TESTE ---")
    print(f"Tamanho do vetor de cada imagem: {len(vetor_anime1)} números.")
    print(f"Distância Euclidiana de Cores/Brilho: {distancia:.4f}")
    
    if distancia == 0:
        print("Resultado: As imagens são idênticas (distância zero)!")
    elif distancia < 0.3:
        print("Resultado: As imagens têm tonalidades e cores MUITO parecidas!")
    else:
        print("Resultado: As imagens possuem paletas de cores bem diferentes.")

Processando imagem do Anime 1...
Processando imagem do Anime 2...

--- RESULTADO DO TESTE ---
Tamanho do vetor de cada imagem: 26 números.
Distância Euclidiana de Cores/Brilho: 0.7784
Resultado: As imagens possuem paletas de cores bem diferentes.


In [18]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from pathlib import Path

# 1. Carregar os Dados do ficheiro CSV (Alterado aqui)
caminho_ficheiro = "banco_de_dados_animes.csv"
df_animes = pd.read_csv(caminho_ficheiro)

# Imprimir as colunas para confirmar os nomes
print("Colunas disponíveis no dataset:", df_animes.columns)

# Definir qual a coluna de texto que será processada
coluna_texto = 'Synopsis' # Substitua pelo nome correto se necessário

# Garantir que não há valores nulos
df_animes[coluna_texto] = df_animes[coluna_texto].fillna("")

# 2. Limpeza de Texto (Texto Limpo)
def limpar_texto(texto):
    texto = texto.lower() # Converter para minúsculas
    texto = re.sub(r'[^\w\s]', '', texto) # Remover pontuação
    texto = re.sub(r'\d+', '', texto) # Remover números (opcional)
    return texto

df_animes['texto_limpo'] = df_animes[coluna_texto].apply(limpar_texto)

# 3. Remoção de Stopwords (Ajustado para Inglês)
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def remover_stopwords(texto):
    palavras = texto.split()
    palavras_filtradas = [palavra for palavra in palavras if palavra not in stop_words]
    return " ".join(palavras_filtradas)

df_animes['texto_sem_stopwords'] = df_animes['texto_limpo'].apply(remover_stopwords)

# 4. Vetorização: Bag of Words (BoW)
vectorizer_bow = CountVectorizer(max_features=1000) # Limitado a 1000 para poupar memória
bow_matrix = vectorizer_bow.fit_transform(df_animes['texto_sem_stopwords'])

# Criar DataFrame com as features do BoW
df_bow = pd.DataFrame(bow_matrix.toarray(), columns=[f"bow_{word}" for word in vectorizer_bow.get_feature_names_out()])
df_final_bow = pd.concat([df_animes, df_bow], axis=1)

# 5. Vetorização: TF-IDF
vectorizer_tfidf = TfidfVectorizer(max_features=2000) # se eu não limitasse isso o dataset do TF-IDF ficava com 78 megas
tfidf_matrix = vectorizer_tfidf.fit_transform(df_animes['texto_sem_stopwords'])

# Criar DataFrame com as features do TF-IDF
df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=[f"tfidf_{word}" for word in vectorizer_tfidf.get_feature_names_out()])
df_final_tfidf = pd.concat([df_animes, df_tfidf], axis=1)

# 6. Guardar os Ficheiros Processados
pasta_dados = Path("/Users/juliano/Projeto1")
pasta_dados.mkdir(exist_ok=True)

# Guardar os dados completos
colunas_exportar = list(df_animes.columns) # Exporta todas as colunas originais + texto limpo
df_animes[colunas_exportar].to_csv(pasta_dados / "animes_completos.csv", index=False)

# Guardar BoW e TF-IDF
df_final_bow.to_csv(pasta_dados / "animes_bow.csv", index=False)
df_final_tfidf.to_csv(pasta_dados / "animes_tfidf.csv", index=False)

print(f"Animes completos: {pasta_dados / 'animes_completos.csv'} (shape={df_animes.shape})")
print(f"Bag of Words:     {pasta_dados / 'animes_bow.csv'}      (shape={df_final_bow.shape})")
print(f"TF-IDF:           {pasta_dados / 'animes_tfidf.csv'}    (shape={df_final_tfidf.shape})")

Colunas disponíveis no dataset: Index(['Title', 'URL', 'Image_URL', 'Synopsis', 'Type', 'Episodes', 'Status',
       'Aired', 'Premiered', 'Broadcast', 'Producers', 'Licensors', 'Studios',
       'Source', 'Genres', 'Demographic', 'Duration', 'Rating', 'Score',
       'Ranked', 'Popularity', 'Members', 'Favorites'],
      dtype='str')


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/juliano/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Animes completos: /Users/juliano/Projeto1/animes_completos.csv (shape=(1050, 25))
Bag of Words:     /Users/juliano/Projeto1/animes_bow.csv      (shape=(1050, 1025))
TF-IDF:           /Users/juliano/Projeto1/animes_tfidf.csv    (shape=(1050, 2025))
